# 0.5 — Auto-label Open Images with a pretrained YOLO

## Purpose

This notebook reads the **raw Open Images COCO export** that notebook 00
produced and uploaded to Drive — the exact same input notebook 01 consumes,
and **NOT** via DVC. It runs a pretrained COCO YOLO over those images to
**add missing bounding boxes** (pseudo-labeling / densification), especially
`food`, which Open Images does not annotate. The enriched COCO will later be
re-uploaded to Drive as a **new version `v2`**, leaving the original `v1`
pristine; notebook 01 will be pointed at `v2` in a later step.

What it does / does NOT do:

- Works **only** on Open Images-sourced data.
- Does **NOT** touch UEC food images or their labels.
- Does **NOT** retrain anything; it only uses a pretrained model for inference.
- Does **NOT** use DVC. The input is the raw COCO export read straight from Drive.

Known limitation: COCO has no flat `plate` class (only `bowl`), so `plate`
densification is inherently bounded by what the COCO model can detect.

## 1. Repository setup

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/LucasGVallejos/iaa-visual-table-assistant.git"
REPO_DIR = Path("/content/iaa-visual-table-assistant")

%cd /content

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already present, pulling latest changes...")
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}

[Errno 2] No such file or directory: '/content'
/Users/alexanderarmua/Projects/UTN/iaa-visual-table-assistant/notebooks
fatal: could not create leading directories of '/content/iaa-visual-table-assistant': Read-only file system
[Errno 2] No such file or directory: '/content/iaa-visual-table-assistant'
/Users/alexanderarmua/Projects/UTN/iaa-visual-table-assistant/notebooks


## 2. Dependencies and GPU check

In [ ]:
!pip install -q -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [ ]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print(
        "[WARN] CUDA is not available in this runtime. YOLO will run on CPU "
        "and inference will be extremely slow. "
        "Switch to a GPU runtime: Runtime > Change runtime type > GPU."
    )

CUDA available: False
[WARN] CUDA is not available in this runtime. YOLO will run on CPU and inference will be extremely slow. Switch to a GPU runtime: Runtime > Change runtime type > GPU.


## 3. Mount Google Drive

Drive holds the raw Open Images zip that notebook 00 produced.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ModuleNotFoundError: No module named 'google.colab'

## 4. Extract and inspect raw Open Images

Extracts the Open Images zip from Drive into
`datasets/raw_datasets/open_images_subset/` (reusing the same helpers
notebook 01 uses) and inspects the COCO export.

> **NOTE — Drive path mismatch.** Notebook 00 uploads to
> `raw_datasets/open_images/`, but this step (like notebook 01) reads from
> `raw_datasets/open_images_subset/`. Ensure the zip lives at the expected
> `open_images_subset/` path on Drive, or move/rename it there before running.

In [ ]:
!python -m src.data.auto_label.prepare_open_images_input --samples 3

## 5. Review a few Open Images samples (current boxes)

Confirms the raw COCO reads and renders correctly before any auto-labeling.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

from src.utils.paths import get_outputs_dir

checks_dir = get_outputs_dir() / "auto_label_checks" / "phase3_raw"
for png in sorted(checks_dir.glob("*.png")):
    display(Image(str(png)))

## 6. Preview pretrained-YOLO detections (read-only)

Loads a pretrained COCO YOLO (`yolov8x.pt`), runs it on a few raw Open Images and prints how each detection maps to our 7 classes (unmapped names are dropped). Nothing is written yet — this is where we eyeball detection quality and the `--conf` threshold before enriching.

In [ ]:
!python -m src.data.auto_label.preview_detections --samples 5 --conf 0.4

In [ ]:
from pathlib import Path
from IPython.display import Image, display

from src.utils.paths import get_outputs_dir

checks_dir = get_outputs_dir() / "auto_label_checks" / "phase5_detections"
for png in sorted(checks_dir.glob("*.png")):
    display(Image(str(png)))

## 7. Enrich a small sample and verify (approval gate)

This is the write step of the auto-labeling workstream, gated behind a
manual eyeball check. The flow is deliberately staged:

1. **Dry-run on 50 images** — counts only, nothing is written. It reports
   how many boxes *would* be added per target class and how many
   detections were dropped (unmapped) or skipped (duplicates/degenerate).
2. **Real 50-image sample** — writes the enriched boxes to a *separate*
   `open_images_subset_v2/labels.json`. The raw v1 export is never
   touched.
3. **Before/after renders** — two-panel PNGs (v1 boxes vs. v1 + the new
   red auto-labeled boxes) so you can judge precision by eye.

Tune `--conf` / `--iou-dedup` here on the small sample before committing to
the full-dataset run (drop `--limit` to process everything).

In [ ]:
!python -m src.data.auto_label.auto_label_open_images --limit 50 --dry-run

In [ ]:
!python -m src.data.auto_label.auto_label_open_images --limit 50

In [ ]:
!python -m src.data.auto_label.verify_autolabel --samples 8

In [ ]:
from pathlib import Path
from IPython.display import Image, display

from src.utils.paths import get_outputs_dir

checks_dir = get_outputs_dir() / "auto_label_checks" / "phase6_verify"
for png in sorted(checks_dir.glob("*.png")):
    display(Image(str(png)))

## 8. Full enrichment run (all images)

This is the production run: it processes **all 13,023 Open Images** with the
pretrained detector and writes the final enriched
`open_images_subset_v2/labels.json` plus the JSON run report. The raw v1
export is never touched. Expect roughly **10–20 minutes on a Colab GPU
(T4)** — do **NOT** run this on CPU (it would take hours). Tune `--conf` /
`--iou-dedup` in section 7 first; the full run uses the same defaults.

In [ ]:
!python -m src.data.auto_label.auto_label_open_images

In [ ]:
import json

from src.utils.paths import get_reports_dir

report_path = get_reports_dir() / "auto_label_report.json"
with open(report_path) as f:
    report = json.load(f)
print(json.dumps(report, indent=2))

In [ ]:
!python -m src.data.auto_label.verify_autolabel --samples 10

In [ ]:
from pathlib import Path
from IPython.display import Image, display

from src.utils.paths import get_outputs_dir

checks_dir = get_outputs_dir() / "auto_label_checks" / "phase6_verify"
for png in sorted(checks_dir.glob("*.png")):
    display(Image(str(png)))

## 9. Package v2 and upload to Drive

This section builds `open_images_table_objects_v2_coco.zip`: the enriched v2
`labels.json` (the original Open Images boxes plus the auto-labeled `Food` and
reinforced cutlery boxes) bundled together with the v1 images it references.
The zip's internal layout mirrors the v1 zip exactly — a root-level
`labels.json` next to `data/<file_name>` entries — so the downstream pipeline
(notebook 01) consumes it **unchanged**.

The upload targets a **new** Drive folder
`raw_datasets/open_images_subset_v2/`, leaving the v1 zip untouched as a
rollback. Run this section **only after** reviewing the section-8 results.

In [ ]:
!python -m src.data.auto_label.package_open_images_v2

In [ ]:
import os, shutil, subprocess
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

REPO = Path("/content/iaa-visual-table-assistant")
if not REPO.exists():
    subprocess.run(["git", "clone",
        "https://github.com/LucasGVallejos/iaa-visual-table-assistant.git", str(REPO)], check=True)
os.chdir(REPO)

V2_LABELS = REPO / "datasets/raw_datasets/open_images_subset_v2/labels.json"
V1_DATA   = REPO / "datasets/raw_datasets/open_images_subset/data"
ZIP       = REPO / "datasets/open_images_table_objects_v2_coco.zip"
BACKUP    = Path("/content/drive/MyDrive/iaa-table-assistant/raw_datasets/labels_v2_full.json")
DRIVE_DIR = Path("/content/drive/MyDrive/iaa-table-assistant/raw_datasets/open_images_subset_v2")

# 1) v2 labels (restore from Drive backup if missing)
if not V2_LABELS.exists():
    V2_LABELS.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BACKUP, V2_LABELS)
    print("[1/4] v2 labels restored from Drive backup")
else:
    print("[1/4] v2 labels OK")

# 2) v1 images (extract from Drive zip if missing)
if not V1_DATA.exists() or not any(V1_DATA.iterdir()):
    print("[2/4] extracting v1 images from Drive (few minutes)...")
    subprocess.run(["python", "-m", "src.data.auto_label.prepare_open_images_input",
                    "--samples", "1"], cwd=REPO, check=True)
else:
    print("[2/4] v1 images OK")

# 3) build zip if missing
if not ZIP.exists():
    print("[3/4] building v2 zip (few minutes)...")
    subprocess.run(["python", "-m", "src.data.auto_label.package_open_images_v2"],
                   cwd=REPO, check=True)
else:
    print("[3/4] zip OK")

# 4) upload to Drive
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
dest = DRIVE_DIR / ZIP.name
print("[4/4] uploading to Drive (few minutes)...")
shutil.copy2(ZIP, dest)
print(f"DONE: {dest} ({dest.stat().st_size / 1024**3:.2f} GB)")


`open_images_subset_v2/` is now the input for notebook 01. A future change by
the author will point `setup_colab_raw_datasets` / notebook 01 at
`open_images_subset_v2/` so the enriched labels flow through the YOLO
conversion. The v1 zip remains on Drive untouched as the rollback.